# Summer Analytics Week 2 Hackathon
## Starter Notebook

This notebook walks you through a simple end-to-end pipeline to get your first submission on the leaderboard. It covers:

- Loading the provided datasets
- Basic preprocessing (handling missing values & encoding)
- Training a **Logistic Regression** baseline model
- Generating predictions for `private_test.csv`
- Creating a valid `submission.csv`

> **Note:** This is just a starting point. Feel free to experiment and improve upon this baseline!


### Step 1: Import Libraries

We start by importing `pandas` for loading and manipulating data, and a few utilities from `scikit-learn` for preprocessing and modelling.
These are all standard libraries that come pre-installed in most Python environments like Kaggle, Colab, or Anaconda.
No additional installations are required to run this notebook.


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


### Step 2: Load Data

We read all three CSV files — `train.csv`, `public_test.csv`, and `private_test.csv` — into pandas DataFrames.
The training set contains labelled examples we will learn from, while the private test set is what we need to make final predictions on.
Printing the shapes gives a quick sanity check that the files loaded correctly.


In [ ]:
train = pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test = pd.read_csv("private_test.csv")

print("Train Shape:", train.shape)
print("Public Test Shape:", public_test.shape)
print("Private Test Shape:", private_test.shape)

train.head()


### Step 3: Define Features and Target

We separate the target column (`Converted`) from the rest of the features in the training set.
For simplicity, we only keep **numeric columns** in this baseline — categorical columns are dropped for now.
This keeps things easy to understand before we add more advanced preprocessing.


In [ ]:
TARGET = "Converted"

# Separate features and target
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]

# Keep only numeric columns for this baseline
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
X_train = X_train[numeric_cols]
X_private = private_test[numeric_cols]

print("Features used:", numeric_cols)


### Step 4: Handle Missing Values & Scale Features

Real-world datasets often have missing values — we fill them in using the **median** of each column, which is robust to outliers.
After that, we apply **Standard Scaling** to bring all features onto the same scale, which is important for Logistic Regression to work well.
We fit both the imputer and scaler only on the training data, then apply the same transformation to the test data to avoid data leakage.


In [ ]:
# Fill missing values with the median of each column
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_private_imputed = imputer.transform(X_private)

# Scale features to zero mean and unit variance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_private_scaled = scaler.transform(X_private_imputed)

print("Preprocessing done! Training data shape:", X_train_scaled.shape)


### Step 5: Train a Logistic Regression Model

Logistic Regression is one of the simplest and most interpretable classification algorithms — a great starting point for any binary classification problem.
It models the probability that a given input belongs to a class using a linear combination of features passed through a sigmoid function.
We set `max_iter=1000` to ensure the solver has enough iterations to converge on this dataset.


In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")
print("Training Accuracy: {:.4f}".format(model.score(X_train_scaled, y_train)))


### Step 6: Generate Predictions and Create Submission

We run the trained model on the scaled private test data to generate our final predictions.
The predictions are combined with the `User_ID` column into a DataFrame matching the required submission format.
The file is saved as `submission.csv` — just upload this to the competition portal to appear on the leaderboard!


In [ ]:
predictions = model.predict(X_private_scaled)

submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
submission.head()
